# 02 — Bag of Words
**Goal:** Convertir texto en vectores numéricos con el modelo Bag-of-Words (BoW).

## La idea central
Ignoramos el orden de las palabras. Un documento se representa por **cuántas veces aparece cada palabra del vocabulario**:

```
vocab    = [app, lenta, rápido, tarjeta, falla]
doc1     = 'la app es lenta'        → [1, 1, 0, 0, 0]
doc2     = 'tarjeta rápido falla'   → [0, 0, 1, 1, 1]
```

El resultado es una **document-term matrix** (DTM): filas = documentos, columnas = palabras.

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
})

In [ ]:
# Dataset de feedback (mismo del notebook 01)
templates_pos = [
    'El proceso de solicitud fue muy rápido y eficiente',
    'Excelente servicio, aprobaron mi tarjeta rápidamente',
    'La app funciona perfecto, muy fácil de usar',
    'Increíbles beneficios, recomiendo la tarjeta al 100',
    'Atención al cliente excepcional, resolvieron todo rápido',
    'Proceso de activación muy sencillo y sin problemas',
]
templates_neg = [
    'La app está muy lenta, tardé mucho en activar la tarjeta',
    'No me llegó el OTP, tuve que llamar varias veces',
    'La carga de documentos falla constantemente',
    'Llevo días esperando la evaluación crediticia sin respuesta',
    'El call center no responde, pésimo servicio al cliente',
    'El proceso de aprobación es muy lento e ineficiente',
]

rng = np.random.default_rng(42)
pos = [templates_pos[rng.integers(6)] for _ in range(300)]
neg = [templates_neg[rng.integers(6)] for _ in range(300)]
comments  = pos + neg
sentiments = [1]*300 + [0]*300

df = pd.DataFrame({'comment': comments, 'sentiment': sentiments})
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
print(f'Dataset: {len(df)} comentarios | {df.sentiment.mean():.0%} positivos')

## 1. BoW a mano — para entender qué hace sklearn

In [ ]:
docs = [
    'la app es lenta y falla',
    'el proceso de solicitud fue rápido',
    'la app es rápida y fácil',
    'el proceso de evaluación falla',
]

# Paso 1: construir vocabulario
all_tokens = set()
for doc in docs:
    all_tokens.update(doc.split())
vocab = sorted(all_tokens)
word2idx = {w: i for i, w in enumerate(vocab)}
print('Vocabulario:', vocab)

# Paso 2: vectorizar cada documento
dtm = np.zeros((len(docs), len(vocab)), dtype=int)
for i, doc in enumerate(docs):
    for word in doc.split():
        dtm[i, word2idx[word]] += 1

df_dtm = pd.DataFrame(dtm, columns=vocab)
df_dtm.index = [f'doc{i}' for i in range(len(docs))]
print('\nDocument-Term Matrix:')
print(df_dtm)

## 2. CountVectorizer — el BoW de sklearn

In [ ]:
# Preprocesador compatible con sklearn
STOPWORDS_ES = [
    'a','al','algo','ante','como','con','de','del','desde','el','ella','en',
    'entre','es','esta','este','esto','fue','la','las','le','lo','los','me',
    'mi','muy','no','nos','o','para','pero','por','que','se','si','sin',
    'su','sus','también','te','todo','un','una','y','ya','yo',
]

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^\wáéíóúüñ\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

vec = CountVectorizer(
    preprocessor=preprocess,
    stop_words=STOPWORDS_ES,
    min_df=2,          # ignorar palabras que aparecen en < 2 docs
    max_df=0.95,       # ignorar palabras que aparecen en > 95% de docs
    ngram_range=(1, 2) # unigramas y bigramas
)

X = vec.fit_transform(df['comment'])   # sparse matrix (600, vocab_size)
print(f'DTM shape: {X.shape}')         # (docs, features)
print(f'Vocabulario: {len(vec.vocabulary_)} tokens')
print(f'Densidad de la matriz: {X.nnz / (X.shape[0]*X.shape[1]):.3%}')

In [ ]:
# Ver las primeras features del vocabulario
feature_names = vec.get_feature_names_out()
print('Primeras 20 features:')
print(feature_names[:20])
print('\nÚltimas 20 features:')
print(feature_names[-20:])

## 3. Análisis de la DTM

In [ ]:
# Frecuencia total de cada palabra en el corpus
total_counts = np.asarray(X.sum(axis=0)).ravel()
top_idx = np.argsort(total_counts)[::-1][:20]

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(feature_names[top_idx][::-1], total_counts[top_idx][::-1], color='#4361ee')
ax.set_xlabel('Frecuencia total en corpus')
ax.set_title('Top 20 tokens por frecuencia')
plt.tight_layout()
plt.show()

In [ ]:
# Frecuencia por clase: positivos vs negativos
y = np.array(sentiments)[df.index]   # resample shuffle above
y = df['sentiment'].to_numpy()

X_pos = X[y == 1]
X_neg = X[y == 0]

freq_pos = np.asarray(X_pos.sum(axis=0)).ravel()
freq_neg = np.asarray(X_neg.sum(axis=0)).ravel()

# Ratio: cuánto más frecuente es en positivos vs negativos
ratio = (freq_pos + 1) / (freq_neg + 1)   # +1 para evitar /0

top_pos_idx = np.argsort(ratio)[::-1][:10]
top_neg_idx = np.argsort(ratio)[:10]

print('Palabras más asociadas a comentarios POSITIVOS:')
for i in top_pos_idx:
    print(f'  {feature_names[i]:30s} ratio={ratio[i]:.1f}x')

print('\nPalabras más asociadas a comentarios NEGATIVOS:')
for i in top_neg_idx:
    print(f'  {feature_names[i]:30s} ratio={ratio[i]:.2f}x')

## 4. Limitaciones del BoW

```
✗ 'no me gustó'  y  'me gustó'  → vectores casi idénticos (ignora orden)
✗ Palabras frecuentes dominan (solución: TF-IDF, próximo notebook)
✗ Sin semántica: 'rápido' y 'veloz' son features distintas
✓ Simple, interpretable, funciona bien para clasificación
```

In [ ]:
# Demostración del problema de negación
neg_vec = CountVectorizer(preprocessor=preprocess, stop_words=None)
neg_vec.fit(['me gustó mucho', 'no me gustó mucho'])

X_neg_demo = neg_vec.transform(['me gustó mucho', 'no me gustó mucho']).toarray()
df_neg = pd.DataFrame(X_neg_demo, columns=neg_vec.get_feature_names_out(),
                       index=['positivo', 'negativo'])
print('BoW no distingue bien la negación:')
print(df_neg)
print('\nSoluciones: bigramas ("no gusto") o modelos con contexto (embeddings)')

## Resumen
| Concepto | Detalle |
|---|---|
| Document-Term Matrix | Filas = docs, columnas = tokens, valores = conteos |
| `CountVectorizer` | BoW de sklearn, acepta `preprocessor`, `stop_words`, `ngram_range` |
| `min_df` / `max_df` | Filtrar tokens demasiado raros o demasiado frecuentes |
| Matriz sparse | Solo guarda valores ≠ 0 — esencial para vocabularios grandes |
| Ratio pos/neg | Señal rápida de qué palabras discriminan clases |

**Siguiente:** `03_tfidf.ipynb` — ponderar palabras por su rareza informativa.